
# Notebook 7: Encoding Categorical Variables

## Objective
Understand how to convert categorical data into numerical
representations suitable for machine learning.

## Learning Outcomes
- Understand nominal, ordinal, and binary variables.
- Apply label, ordinal, one-hot, and dummy encoding.
- Understand frequency and target encoding.
- Handle unknown and high-cardinality categories.
- Select encoding methods appropriate to each problem.
- Understand the risks of inappropriate encoding.


In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    LabelEncoder,
    OrdinalEncoder,
    OneHotEncoder
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
data = {
    "Employee_ID": [101, 102, 103, 104, 105, 106, 107, 108],
    "Department": [
        "IT", "HR", "Finance", "IT",
        "HR", "Finance", "IT", "Marketing"
    ],
    "Education": [
        "Bachelor", "Master", "PhD", "Bachelor",
        "Master", "PhD", "Bachelor", "Master"
    ],
    "Experience_Level": [
        "Junior", "Mid", "Senior", "Junior",
        "Mid", "Senior", "Mid", "Junior"
    ],
    "Remote_Work": [
        "Yes", "No", "Yes", "No",
        "Yes", "No", "Yes", "No"
    ],
    "Salary": [35000, 42000, 65000, 38000, 45000, 70000, 40000, 43000]
}

df_encoding = pd.DataFrame(data)

display(df_encoding)

,Employee_ID,Department,Education,Experience_Level,Remote_Work,Salary
0,101,IT,Bachelor,Junior,Yes,35000
1,102,HR,Master,Mid,No,42000
2,103,Finance,PhD,Senior,Yes,65000
3,104,IT,Bachelor,Junior,No,38000
4,105,HR,Master,Mid,Yes,45000
5,106,Finance,PhD,Senior,No,70000
6,107,IT,Bachelor,Mid,Yes,40000
7,108,Marketing,Master,Junior,No,43000



## 1. What Is Categorical Encoding?

Categorical encoding converts categories or labels into numerical
representations that machine learning algorithms can process.

Examples:
- Department: IT, HR, Finance
- Education: Bachelor, Master, PhD
- Remote_Work: Yes, No

Different encoding methods suit different types of categories.
The method should preserve meaningful relationships without
introducing misleading numerical information.



## 2. Types of Categorical Variables

### Nominal Variables
Categories without a natural order, such as department or city.

### Ordinal Variables
Categories with a meaningful order, such as Junior, Mid, and Senior.

### Binary Variables
Variables with two possible categories, such as Yes and No.

The variable type determines which encoding method is appropriate.



In [3]:
label_encoder = LabelEncoder()

df_encoding["Department_Label"] = label_encoder.fit_transform(
    df_encoding["Department"]
)

display(df_encoding[["Department", "Department_Label"]])

,Department,Department_Label
0,IT,2
1,HR,1
2,Finance,0
3,IT,2
4,HR,1
5,Finance,0
6,IT,2
7,Marketing,3


In [4]:
education_order = ["Bachelor", "Master", "PhD"]

ordinal_encoder = OrdinalEncoder(
    categories=[education_order],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

df_encoding["Education_Ordinal"] = ordinal_encoder.fit_transform(
    df_encoding[["Education"]]
).ravel().astype(int)

display(df_encoding[["Education", "Education_Ordinal"]])

,Education,Education_Ordinal
0,Bachelor,0
1,Master,1
2,PhD,2
3,Bachelor,0
4,Master,1
5,PhD,2
6,Bachelor,0
7,Master,1


In [5]:
one_hot = pd.get_dummies(
    df_encoding["Department"],
    prefix="Department",
    dtype=int
)

display(one_hot)

,Department_Finance,Department_HR,Department_IT,Department_Marketing
0,0,0,1,0
1,0,1,0,0
2,1,0,0,0
3,0,0,1,0
4,0,1,0,0
5,1,0,0,0
6,0,0,1,0
7,0,0,0,1


In [6]:
dummy_encoded = pd.get_dummies(
    df_encoding["Department"],
    prefix="Department",
    drop_first=True,
    dtype=int
)

display(dummy_encoded)

,Department_HR,Department_IT,Department_Marketing
0,0,1,0
1,1,0,0
2,0,0,0
3,0,1,0
4,1,0,0
5,0,0,0
6,0,1,0
7,0,0,1


In [7]:
binary_mapping = {"No": 0, "Yes": 1}

df_encoding["Remote_Work_Binary"] = (
    df_encoding["Remote_Work"].map(binary_mapping)
)

display(df_encoding[["Remote_Work", "Remote_Work_Binary"]])

,Remote_Work,Remote_Work_Binary
0,Yes,1
1,No,0
2,Yes,1
3,No,0
4,Yes,1
5,No,0
6,Yes,1
7,No,0


In [8]:
department_frequency = df_encoding["Department"].value_counts()

df_encoding["Department_Frequency"] = (
    df_encoding["Department"].map(department_frequency)
)

display(df_encoding[["Department", "Department_Frequency"]])

,Department,Department_Frequency
0,IT,3
1,HR,2
2,Finance,2
3,IT,3
4,HR,2
5,Finance,2
6,IT,3
7,Marketing,1


In [9]:
target_demo = df_encoding[["Department", "Salary"]].copy()

global_mean = target_demo["Salary"].mean()

department_means = target_demo.groupby(
    "Department"
)["Salary"].mean()

target_demo["Department_Target_Encoded"] = (
    target_demo["Department"]
    .map(department_means)
)

display(target_demo)

,Department,Salary,Department_Target_Encoded
0,IT,35000,37666.666667
1,HR,42000,43500.000000
2,Finance,65000,67500.000000
3,IT,38000,37666.666667
4,HR,45000,43500.000000
5,Finance,70000,67500.000000
6,IT,40000,37666.666667
7,Marketing,43000,43000.000000


In [10]:
train_categories = pd.DataFrame({
    "Department": ["IT", "HR", "Finance", "IT"]
})

new_categories = pd.DataFrame({
    "Department": ["IT", "Marketing"]
})

unknown_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

train_encoded = unknown_encoder.fit_transform(train_categories)
new_encoded = unknown_encoder.transform(new_categories)

print("Known categories:")
display(pd.DataFrame(
    train_encoded,
    columns=unknown_encoder.get_feature_names_out()
))

print("\nNew data with an unknown category:")
display(pd.DataFrame(
    new_encoded,
    columns=unknown_encoder.get_feature_names_out()
))

Known categories:


,Department_Finance,Department_HR,Department_IT
0,0.0,0.0,1.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0



New data with an unknown category:


,Department_Finance,Department_HR,Department_IT
0,0.0,0.0,1.0
1,0.0,0.0,0.0


In [11]:
high_cardinality = pd.DataFrame({
    "Postal_Code": [
        "500001", "500002", "500003",
        "500001", "500004", "500005"
    ]
})

print("Number of unique postal codes:")
print(high_cardinality["Postal_Code"].nunique())

print("\nFrequency encoding:")
high_cardinality["Postal_Code_Frequency"] = (
    high_cardinality["Postal_Code"]
    .map(high_cardinality["Postal_Code"].value_counts())
)

display(high_cardinality)

Number of unique postal codes:
5

Frequency encoding:


,Postal_Code,Postal_Code_Frequency
0,500001,2
1,500002,1
2,500003,1
3,500001,2
4,500004,1
5,500005,1



## 3. Choosing an Encoding Method

| Method | Best use | Main risk |
|---|---|---|
| Label encoding | Target labels | Artificial ordering in input features |
| Ordinal encoding | Ordered categories | Incorrect or misleading order |
| One-hot encoding | Nominal categories | Feature expansion |
| Dummy encoding | Nominal categories, often linear models | Reference-category interpretation |
| Binary mapping | Two-category variables | Incorrect or inconsistent mapping |
| Frequency encoding | Some high-cardinality features | Loss of category identity |
| Target encoding | High-cardinality predictive features | Target leakage and overfitting |

Encoding decisions should reflect the feature's meaning, the
algorithm, dataset size, and the validation strategy.



## Conclusion

In this notebook, I learned how to identify nominal, ordinal,
and binary categorical variables and encode them using different
techniques.

I practiced label encoding, ordinal encoding, one-hot encoding,
dummy encoding, binary mapping, frequency encoding, and target encoding.

I also learned to handle unknown categories and understand
high-cardinality features.

Inappropriate encoding can introduce false ordering, increase
dimensionality, lose information, or cause target leakage.
The encoding method must be selected according to the data and
the machine learning task.
